# 20 · Structured output: from "please" to JSON Schema
Level L2 · Part 3 · core · ~30 min · builds on 19

## Goal
Get a `{title, topics, date}` record for Scout's notes from Gemini in four ways (ask nicely, JSON mode,
a JSON Schema, a Pydantic parse helper), score each with the same checker, and say what each one
guarantees and what it does not.

## The idea
Scout's code needs data, not prose: a dict it can store and search. You can **ask** for JSON in the
prompt, but nothing stops the model from adding a sentence or a code fence. **JSON mode** tells the API
"only valid JSON", but any keys. A **JSON Schema** goes further: with **constrained decoding** the server
only lets the model pick tokens that keep the output inside the schema. A **parse helper** takes a
Python class, sends its schema for you and hands back an object. Each step moves a guarantee from
"the model tried" to "the API enforces it". None of them checks that the values are *true*, like a
paper form whose DD/MM/YYYY boxes fix the shape of a birth date but not whether it is honest.

## Picture
```mermaid
flowchart LR
    A["1 ask in prompt<br/>guarantee: none"] --> B["2 JSON mode<br/>valid JSON"]
    B --> C["3 JSON Schema<br/>valid JSON + shape"]
    C --> D["4 parse helper<br/>shape + Python object"]
    D --> V["your checks<br/>(values, meaning)"]
```

## Step by step
### 1. Start the recorder and set up
Same as lesson 19: the recorder first, then the notes, client and pinned model. Without a key the calls
replay from `../data/llm_cassettes/20_structured-output.yaml`.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("20_structured-output")

LLM mode: replay (20_structured-output.yaml)


Setup from lesson 07: the notes, client and model id, unchanged. We test on three notes; note 5 is the
only one that mentions a date.

In [2]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)
TEST_NOTES = [NOTES[0], NOTES[2], NOTES[4]]
print([n["id"] for n in TEST_NOTES], "| model:", MODEL)

[1, 3, 5] | model: gemini-3.5-flash-lite


### 2. Write the checker first
As in lesson 19, decide what "usable" means before calling anything. Scout stores a record only if the
reply parses as JSON and has exactly `title` (text), `topics` (list of text) and `date` (`"YYYY-MM"` or
`null`).

In [3]:
import json
import re

def check_record(text: str) -> list[str]:
    """Return the rules a reply breaks (empty list = Scout can store it)."""
    try:
        obj = json.loads(text or "")
    except json.JSONDecodeError as e:
        return [f"not JSON ({e.msg})"]
    if not isinstance(obj, dict):
        return ["not a JSON object"]
    problems = []
    if set(obj) != {"title", "topics", "date"}:
        problems.append(f"keys are {sorted(obj)}")
    if not isinstance(obj.get("title"), str):
        problems.append("title is not text")
    if not (isinstance(obj.get("topics"), list) and all(isinstance(t, str) for t in obj["topics"])):
        problems.append("topics is not a list of text")
    d = obj.get("date")
    if d is not None and not (isinstance(d, str) and re.fullmatch(r"\d{4}-\d{2}", d)):
        problems.append(f"date {d!r} is not YYYY-MM")
    return problems

for sample in ['{"title": "Moons", "topics": ["mars"], "date": null}', 'Sure! {"title": "Moons"}']:
    print(f"{sample!r:58} -> {check_record(sample) or 'ok'}")

'{"title": "Moons", "topics": ["mars"], "date": null}'     -> ok
'Sure! {"title": "Moons"}'                                 -> ['not JSON (Expecting value)']


One helper runs a config on the three test notes and scores the replies, so the four ways differ only
in what we pass to it.

In [4]:
def run(make_prompt, config: types.GenerateContentConfig) -> list:
    replies, passed = [], 0
    for n in TEST_NOTES:
        resp = client.models.generate_content(model=MODEL, contents=make_prompt(n), config=config)
        replies.append(resp)
        problems = check_record(resp.text)
        passed += not problems
        print(f"note {n['id']}: {(resp.text or '').strip()!r}")
        print(f"         -> {'PASS' if not problems else 'FAIL ' + ', '.join(problems)}")
    print(f"=> {passed}/{len(TEST_NOTES)} storable records")
    return replies

### 3. Way 1: ask in the prompt
First the prompt most people write on day one. The API is told nothing about JSON; only the words ask.

In [5]:
def casual_prompt(note: dict) -> str:
    return f"Extract the title, topics and date from this note as JSON: {note['text']}"

PLAIN = types.GenerateContentConfig(max_output_tokens=200, automatic_function_calling=NO_AFC)
print(casual_prompt(NOTES[0]))

Extract the title, topics and date from this note as JSON: Mars has two small moons, Phobos and Deimos.


Run it on the three notes and score the replies.

In [6]:
way1_casual = run(casual_prompt, PLAIN)

note 1: '```json\n{\n  "title": null,\n  "topics": [\n    "Mars",\n    "Moons",\n    "Phobos",\n    "Deimos"\n  ],\n  "date": null\n}\n```'
         -> FAIL not JSON (Expecting value)
note 3: '```json\n{\n  "title": null,\n  "topics": ["Mars", "Astronomy", "Planetary Science"],\n  "date": null\n}\n```'
         -> FAIL not JSON (Expecting value)
note 5: '```json\n{\n  "title": "Perseverance Landing on Mars",\n  "topics": [\n    "Perseverance",\n    "Mars",\n    "Rover",\n    "Space Exploration"\n  ],\n  "date": "February 2021"\n}\n```'
         -> FAIL not JSON (Expecting value)
=> 0/3 storable records


Now the lesson 19 way: data first, then instructions that name every key and say "JSON only".

In [7]:
ASK = """Reply with JSON only: an object with
"title" (a short title, under 8 words), "topics" (a list of 1 to 3 lowercase topic words),
"date" (the month the note mentions, as YYYY-MM, or null if it mentions none)."""

def ask_prompt(note: dict) -> str:
    return f"<note>{note['text']}</note>\n\n<instructions>\n{ASK}\n</instructions>"

print(ask_prompt(NOTES[4]))

<note>The rover Perseverance landed on Mars in February 2021.</note>

<instructions>
Reply with JSON only: an object with
"title" (a short title, under 8 words), "topics" (a list of 1 to 3 lowercase topic words),
"date" (the month the note mentions, as YYYY-MM, or null if it mentions none).
</instructions>


Same notes, same config, only the prompt changed.

In [8]:
way1 = run(ask_prompt, PLAIN)

note 1: '{\n  "title": "Moons of Mars",\n  "topics": [\n    "mars",\n    "astronomy",\n    "moons"\n  ],\n  "date": null\n}'
         -> PASS
note 3: '{\n  "title": "Length of a Martian Day",\n  "topics": [\n    "mars",\n    "astronomy",\n    "time"\n  ],\n  "date": null\n}'
         -> PASS
note 5: '```json\n{\n  "title": "Perseverance Mars Landing",\n  "topics": [\n    "mars",\n    "rover",\n    "space"\n  ],\n  "date": "2021-02"\n}\n```'
         -> FAIL not JSON (Expecting value)
=> 2/3 storable records


The casual prompt got 0/3: every reply came wrapped in a Markdown code fence (` ```json `), which
`json.loads` rejects, and two titles were `null`. The careful prompt did much better, yet note 5 still
came back fenced: 2/3. A good prompt raises the odds; it cannot *force* JSON, because the request never
told the API we wanted JSON.

### 4. Way 2: JSON mode
Back to the casual prompt, plus one config field: `response_mime_type="application/json"`. The API now
returns only JSON text. It still knows nothing about *which* keys or formats we want.

In [9]:
JSON_MODE = types.GenerateContentConfig(max_output_tokens=200, automatic_function_calling=NO_AFC,
                                        response_mime_type="application/json")
print("response_mime_type:", JSON_MODE.response_mime_type)

response_mime_type: application/json


The casual prompt again, now with JSON mode on.

In [10]:
way2 = run(casual_prompt, JSON_MODE)

note 1: '{\n  "title": "Mars Moons",\n  "topics": [\n    "Mars",\n    "Phobos",\n    "Deimos",\n    "Astronomy"\n  ],\n  "date": null\n}'
         -> PASS
note 3: '{\n  "title": "A day on Mars",\n  "topics": [\n    "Mars",\n    "Astronomy",\n    "Time"\n  ],\n  "date": null\n}'
         -> PASS
note 5: '{\n  "title": "Rover Perseverance Mars Landing",\n  "topics": [\n    "Rover Perseverance",\n    "Mars",\n    "Space Exploration"\n  ],\n  "date": "February 2021"\n}'
         -> FAIL date 'February 2021' is not YYYY-MM
=> 2/3 storable records


No fences now: all three replies parse. But with nothing telling it the format, the model wrote note 5's
date as `"February 2021"`, and topics in Title Case. JSON mode fixed the *syntax*; the *shape* is still
the model's guess.

### 5. How the API can enforce a shape: constrained decoding
Lesson 02 showed the model picks one token at a time from a probability list. With a schema, the server
first removes every token that would break the schema, then samples from what is left. A toy version,
with made-up probabilities, at the point where the reply so far is `{"date": `:

In [11]:
next_token_probs = {'"Feb': 0.45, '"2021': 0.30, "null": 0.15, "Sure": 0.10}  # toy numbers

def allowed(token: str) -> bool:
    # schema says: date is null or a string of the form "YYYY-MM"
    return token == "null" or re.fullmatch(r'"\d{0,4}', token) is not None

kept = {t: p for t, p in next_token_probs.items() if allowed(t)}
total = sum(kept.values())
print("masked out:", [t for t in next_token_probs if t not in kept])
print("renormalised:", {t: round(p / total, 2) for t, p in kept.items()})

masked out: ['"Feb', 'Sure']
renormalised: {'"2021': 0.67, 'null': 0.33}


`"Feb` was the model's favourite, but it can never be chosen. That is why a schema is a *guarantee* and
a prompt is a *request*. Real servers compile the schema into a grammar and apply this mask to the
whole vocabulary at every step.

### 6. Way 3: a JSON Schema
We write the shape as a **JSON Schema** dict and pass it as `response_json_schema` (JSON mode stays on).
The `description` fields carry the rules a schema cannot express, so the prompt shrinks to the note and
one line.

In [12]:
NOTE_SCHEMA = {
    "type": "object",
    "properties": {
        "title": {"type": "string", "description": "short title, under 8 words"},
        "topics": {"type": "array", "items": {"type": "string"}, "minItems": 1, "maxItems": 3,
                   "description": "lowercase topic words"},
        "date": {"anyOf": [{"type": "string"}, {"type": "null"}],
                 "description": "month the note mentions, YYYY-MM, else null"},
    },
    "required": ["title", "topics", "date"],
    "additionalProperties": False,
}

def short_prompt(note: dict) -> str:
    return f"<note>{note['text']}</note>\n\nExtract a record for this note."

SCHEMA = types.GenerateContentConfig(max_output_tokens=200, automatic_function_calling=NO_AFC,
                                     response_mime_type="application/json", response_json_schema=NOTE_SCHEMA)
print(short_prompt(NOTES[4]))

<note>The rover Perseverance landed on Mars in February 2021.</note>

Extract a record for this note.


No format instructions in the prompt at all; the schema carries them.

In [13]:
way3 = run(short_prompt, SCHEMA)

note 1: '{"title": "Moons of Mars", "topics": ["mars", "space", "astronomy"], "date": null}'
         -> PASS
note 3: '{\n  "title": "Length of a Martian day",\n  "topics": [\n    "mars",\n    "astronomy",\n    "time"\n  ],\n  "date": null\n}'
         -> PASS
note 5: '{\n  "title": "Perseverance Lands on Mars",\n  "topics": [\n    "rover",\n    "mars",\n    "space"\n  ],\n  "date": "2021-02"\n}'
         -> PASS
=> 3/3 storable records


3/3, with a one-line prompt. The keys and types now come from the API, not from the prompt, and the
descriptions steered the values: lowercase topics, `"2021-02"`. Be precise about what is enforced,
though: the schema says "string or null", while `YYYY-MM` lives only in a description, so that part is
still a request. It held here; the checker is what tells you whether it always does.

### 7. Way 4: a parse helper with a Pydantic class
Writing schema dicts by hand gets old. A **parse helper** takes a Python class: pass a Pydantic
`BaseModel` as `response_schema` and the SDK turns it into a schema, sends it, and puts a ready object
in `resp.parsed`. (Lesson 21 covers Pydantic validation in depth; here the class is just the shape.)

In [14]:
from pydantic import BaseModel, Field

class NoteRecord(BaseModel):
    title: str = Field(description="short title, under 8 words")
    topics: list[str] = Field(description="1 to 3 lowercase topic words")
    date: str | None = Field(description="month the note mentions, YYYY-MM, else null")

PARSED = types.GenerateContentConfig(max_output_tokens=200, automatic_function_calling=NO_AFC,
                                     response_mime_type="application/json", response_schema=NoteRecord)
print(json.dumps(NoteRecord.model_json_schema()["properties"]["date"]))

{"anyOf": [{"type": "string"}, {"type": "null"}], "description": "month the note mentions, YYYY-MM, else null", "title": "Date"}


The printed piece is the schema the SDK sends for `date`: `str | None` became an `anyOf`. Now run it.

In [15]:
way4 = run(short_prompt, PARSED)

note 1: '{\n  "title": "Moons of Mars",\n  "topics": [\n    "mars",\n    "astronomy",\n    "moons"\n  ],\n  "date": null\n}'
         -> PASS
note 3: '{"title": "Length of a Mars Day", "topics": ["mars", "astronomy", "time"], "date": null}'
         -> PASS
note 5: '{\n  "title": "Perseverance lands on Mars",\n  "topics": [\n    "mars",\n    "rover",\n    "space"\n  ],\n  "date": "2021-02"\n}'
         -> PASS
=> 3/3 storable records


The text was scored as before. The new part is `resp.parsed`: a `NoteRecord`, so Scout uses attributes,
not dict keys, and an editor can autocomplete them.

In [16]:
records = [r.parsed for r in way4]
for n, rec in zip(TEST_NOTES, records):
    print(f"note {n['id']}: {type(rec).__name__} title={rec.title!r} topics={rec.topics} date={rec.date}")

note 1: NoteRecord title='Moons of Mars' topics=['mars', 'astronomy', 'moons'] date=None
note 3: NoteRecord title='Length of a Mars Day' topics=['mars', 'astronomy', 'time'] date=None
note 5: NoteRecord title='Perseverance lands on Mars' topics=['mars', 'rover', 'space'] date=2021-02


### 8. What each way guarantees
Count the results side by side, then read the table: it says what is *promised*, which matters more
than one lucky run.

In [17]:
ways = {"1 ask (casual)": way1_casual, "1 ask (careful)": way1, "2 JSON mode": way2,
        "3 JSON Schema": way3, "4 parse helper": way4}
for name, replies in ways.items():
    ok = sum(not check_record(r.text) for r in replies)
    print(f"{name:16} {ok}/{len(replies)} storable")

1 ask (casual)   0/3 storable
1 ask (careful)  2/3 storable
2 JSON mode      2/3 storable
3 JSON Schema    3/3 storable
4 parse helper   3/3 storable


| Way | Valid JSON | Right keys and types | Values correct | You get |
|---|---|---|---|---|
| 1 ask in prompt | no promise | no promise | no | text |
| 2 JSON mode | yes | no promise | no | text |
| 3 JSON Schema | yes | yes (within the vendor's schema subset) | no | text |
| 4 parse helper | yes | yes | no | object |

Two holes stay open in every row. A reply cut off by `max_output_tokens` (lesson 13) or replaced by a
safety refusal can break even a schema, so read the finish reason. And "Values correct" is always "no":
lesson 21 validates values and retries.

### 9. Same idea, every vendor, different limits
All major APIs take a JSON Schema now. The field names differ. These request shapes are built here,
not sent: we have no Anthropic/OpenAI/Ollama setup in this course yet, and the Gemini Interactions API
is not reachable on our Vertex express-mode key (it returns 404). The course uses `generate_content`,
so the Interactions row is read-along.

In [18]:
vendor_requests = {
    "Gemini generate_content": {"config": {"response_mime_type": "application/json", "response_json_schema": "<schema>"}},
    "Gemini Interactions":     {"response_format": {"type": "text", "mime_type": "application/json", "schema": "<schema>"}},
    "OpenAI Responses":        {"text": {"format": {"type": "json_schema", "name": "note_record", "strict": True, "schema": "<schema>"}}},
    "Anthropic Messages":      {"output_config": {"format": {"type": "json_schema", "schema": "<schema>"}}},
    "Ollama (local)":          {"format": "<schema>"},
}
for vendor, shape in vendor_requests.items():
    print(f"{vendor:24} {json.dumps(shape)}")

Gemini generate_content  {"config": {"response_mime_type": "application/json", "response_json_schema": "<schema>"}}
Gemini Interactions      {"response_format": {"type": "text", "mime_type": "application/json", "schema": "<schema>"}}
OpenAI Responses         {"text": {"format": {"type": "json_schema", "name": "note_record", "strict": true, "schema": "<schema>"}}}
Anthropic Messages       {"output_config": {"format": {"type": "json_schema", "schema": "<schema>"}}}
Ollama (local)           {"format": "<schema>"}


Each vendor supports only a **schema subset**, and the subsets differ. From the docs (RESEARCH.md):
Gemini lists the keywords it accepts and has no `minLength`/`maxLength`; Anthropic rejects number and
string limits and array limits beyond `minItems` 0/1 (our linter simply flags every `maxItems`), and needs `additionalProperties: false` on every object; OpenAI strict
mode needs every property in `required` and `additionalProperties: false`. A small linter catches
this before you deploy.

In [19]:
GEMINI_OK = {"$id", "$defs", "$ref", "$anchor", "type", "format", "title", "description", "enum", "items",
             "prefixItems", "minItems", "maxItems", "minimum", "maximum", "anyOf", "oneOf", "properties",
             "additionalProperties", "required", "propertyOrdering"}
ANTHROPIC_BAD = {"minimum", "maximum", "multipleOf", "minLength", "maxLength", "maxItems"}

def lint(schema: dict, vendor: str, path: str = "$") -> list[str]:
    issues = []
    for key, value in schema.items():
        if vendor == "gemini" and key not in GEMINI_OK:
            issues.append(f"{path}: '{key}' not supported")
        if vendor == "anthropic" and key in ANTHROPIC_BAD:
            issues.append(f"{path}: '{key}' not supported")
    if vendor == "anthropic" and schema.get("type") == "object" and schema.get("additionalProperties") is not False:
        issues.append(f"{path}: object needs additionalProperties: false")
    for name, sub in schema.get("properties", {}).items():
        issues += lint(sub, vendor, f"{path}.{name}")
    if isinstance(schema.get("items"), dict):
        issues += lint(schema["items"], vendor, f"{path}[]")
    return issues

Run it on our schema, and on a version that adds a `maxLength` to the title.

In [20]:
import copy
STRICTER = copy.deepcopy(NOTE_SCHEMA)
STRICTER["properties"]["title"]["maxLength"] = 60
for vendor in ["gemini", "anthropic"]:
    print(f"{vendor:9} NOTE_SCHEMA: {lint(NOTE_SCHEMA, vendor) or 'ok'}")
    print(f"{vendor:9} STRICTER:    {lint(STRICTER, vendor) or 'ok'}")

gemini    NOTE_SCHEMA: ok
gemini    STRICTER:    ["$.title: 'maxLength' not supported"]
anthropic NOTE_SCHEMA: ["$.topics: 'maxItems' not supported"]
anthropic STRICTER:    ["$.title: 'maxLength' not supported", "$.topics: 'maxItems' not supported"]


The same schema is fine on one vendor and refused on another. The SDK helpers (OpenAI `parse`,
Anthropic `messages.parse`) quietly rewrite unsupported keywords into descriptions and check them after
the reply instead. The Try-it builds that rewrite.

## At work
- **Schema-constrained output replaced "parse and pray".** Code that forced JSON by prefilling `{`
  broke when Claude 4.6+ started rejecting prefill with a 400; the replacement is `output_config.format`
  or a parse helper. → lesson 19 (prefill), lesson 21 (what to do when values are still wrong)
- **Validate even with a schema.** Teams still run Pydantic on every reply, retry with the validation
  error fed back, cap retries at 2 or 3, and log the failure rate as a metric. → lesson 21
- **Schemas fail at deploy time, per vendor.** Anthropic limits a request to 24 optional parameters and
  16 union-typed ones, and compiles each new schema into a grammar (slow first call, cached 24 hours).
  Switching vendor means re-linting every schema. → lesson 24 (strict schemas for tools)
- **A refusal or a cut-off still breaks the shape.** Production code checks the stop reason before it
  trusts the JSON. → lesson 13 (stop reasons), lesson 49 (logging the failures)

## What just happened
- We wrote `check_record` first and ran four ways of getting a record for the same three notes.
- Asking in the prompt gives no guarantee: 0/3 with a casual prompt, 2/3 with a careful one (a fence
  slipped through). JSON mode fixed the syntax but not the shape: 2/3, a date came back as words.
- A JSON Schema got 3/3 with a one-line prompt: the server masks tokens that would break it (as the
  toy mask showed), so keys and types are guaranteed.
- A Pydantic class as `response_schema` gives the same guarantee plus a typed object in `resp.parsed`.
- Rules that live only in a `description` (like `YYYY-MM`) are still requests, and no way checks truth.
- Every vendor has its own field names and its own schema subset; a linter catches the gaps early.

## Common mistakes
**1. Calling `json.loads` on a reply you only asked for.** The casual replies from step 3 are wrapped
in a code fence; the first one crashes the loop that stores them.

In [21]:
stored = [json.loads(r.text) for r in way1_casual]

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

Fix: turn on JSON mode or, better, a schema, so the API returns JSON. Way 3's replies load cleanly.

In [22]:
stored = [json.loads(r.text) for r in way3]
print(stored[0])

{'title': 'Moons of Mars', 'topics': ['mars', 'space', 'astronomy'], 'date': None}


**2. Trusting `resp.parsed` blindly.** The SDK sets `parsed` to `None`, without an error, when the text
does not fit the class (for example a reply cut off by `max_output_tokens`). We copy the SDK's parsing
logic and feed it a truncated reply.

In [23]:
from pydantic import ValidationError

def sdk_parse(text: str):          # what google-genai does to fill resp.parsed
    try:
        return NoteRecord.model_validate_json(text)
    except ValidationError:
        return None                # no error raised: parsed is just None

parsed = sdk_parse(way4[0].text[:25])   # the first 25 characters: a reply cut short
print("title:", parsed.title)

AttributeError: 'NoneType' object has no attribute 'title'

Fix: check `parsed` (and the finish reason) before using it, and treat `None` as a failed call.

In [24]:
def record_or_none(resp):
    reason = resp.candidates[0].finish_reason
    if resp.parsed is None or reason != types.FinishReason.STOP:
        print(f"unusable reply (finish_reason={reason}, parsed={resp.parsed})")
        return None
    return resp.parsed

print(record_or_none(way4[0]))
# a stub reply with nothing parsed (no model call) shows the failure path
print(record_or_none(type("R", (), {"parsed": None, "candidates": way4[0].candidates})()))

title='Moons of Mars' topics=['mars', 'astronomy', 'moons'] date=None
unusable reply (finish_reason=FinishReason.STOP, parsed=None)
None


## Try it
Write `to_anthropic(schema)`: return a copy of a schema that `lint(..., "anthropic")` accepts, the way
the SDK helpers do it. Remove each keyword in `ANTHROPIC_BAD`, append it to that property's
`description` (for example `"(maxLength: 60)"`), and set `additionalProperties: false` on every object.
Test it on `STRICTER`.

In [25]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [26]:
llm_replay.stop()

## Key terms
- **structured output** — model output in a machine-readable shape (here JSON) that code can use directly.
- **JSON mode** — an API setting (`response_mime_type="application/json"` on Gemini) that guarantees valid JSON but not which keys or types it holds.
- **JSON Schema** — a standard JSON format for describing the shape of JSON data: types, properties, required keys, allowed values.
- **constrained decoding** — the server removes, at each step, every token that would break the schema before sampling, so the output always fits it.
- **parse helper** — an SDK feature that takes a class (e.g. a Pydantic model), sends its schema and returns a typed object (`resp.parsed` in google-genai).
- **schema subset** — the part of JSON Schema a vendor's structured output supports; keywords outside it are rejected or ignored, and the subset differs per vendor.